# 02 · 用 Claude Agent SDK 組合 Agentic RAG 架構

> 前置：`01_agent_sdk_basics.ipynb`
> 這本會實際跑 **Naive / RAG-Fusion / HyDE / CRAG / Self-RAG / Adaptive-RAG**，最後你自己組一個。

## 這本要打掉一個迷思

學 RAG 的人最痛苦的是：論文一篇一篇冒出來，CRAG、Self-RAG、Adaptive-RAG、RAPTOR、HyDE……
看起來每個都要重學一次。

**Modular RAG（Gao et al. 2024）的主張是：它們不是六個系統，是同一組模組的六種編排。**

```
       模組（能力）                          編排（政策）
  ┌──────────────────────┐         ┌──────────────────────────┐
  │ Pre-retrieval        │         │ Linear      固定順序跑完   │
  │   multi_search       │         │ Conditional 依判斷走分支   │
  │   hyde_search        │    ×    │ Branching   展開多路再合併 │
  │ Retrieval            │         │ Looping     生成→批判→重來 │
  │   search             │         └──────────────────────────┘
  │ Post-retrieval       │
  │   grade_documents    │              = 一個具名架構
  │   expand             │
  │   diversify          │
  └──────────────────────┘
```

在 Claude Agent SDK 裡，這個對應非常乾淨：

| Modular RAG 概念 | SDK 裡是什麼 |
|---|---|
| 模組（module） | 一個 `@tool` |
| 編排（orchestration） | `system_prompt` 裡的流程規則 |
| 架構（architecture） | 模組清單 + 編排規則 |

> **所以「換架構」不是改程式碼，是換一份設定。** 這本 notebook 就是在證明這句話。

In [1]:
import os, sys
from pathlib import Path

# 跟 01 一樣：把工作目錄切到專案根，才 import 得到 retrieval.py / modules.py
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

from dotenv import load_dotenv
load_dotenv()                       # 讀 .env 裡的認證設定進環境變數

# kernel 選對了嗎？VS Code 打開 notebook 常會自動建一個空的 notebooks/.venv，裡面只有 ipykernel，
# 什麼都 import 不到。這裡先試 import，失敗就直接告訴你怎麼修，不要讓你對著 ModuleNotFoundError 猜。
try:
    from retrieval import load_index    # 檢索層：載入索引、BM25、向量、RRF 都在這
except ModuleNotFoundError:
    print(f"這個 kernel 沒裝專案的套件（它用的 Python 是 {sys.executable}）。")
    print("notebook 要用專案根目錄的 .venv —— 也就是 `uv sync --extra embeddings` 裝出來的那個。")
    print("VS Code：右上角 Select Kernel → Python Environments → 選 agentic-rag-workshop/.venv（Python 3.13）")
    raise
import modules as M                 # 模組層：七個 @tool、Architecture、arun、compare 都在這。後面一律叫它 M

# 知識庫 = corpus/ 裡那份 Claude Code 參考文件，已經事先切成片段、建好索引放在 data/。
# 還沒建索引的話先到終端機跑：
#   uv run python index_corpus.py --no-vectors     （純 BM25，秒建）
#   uv run python index_corpus.py                  （含向量，第一次會下載約 100MB 模型）
# 要換成自己的知識庫：uv run python index_corpus.py --root ~/你的資料夾
#
# load_index() 回傳一個 Index 物件，整本 notebook 叫它 ix，裡面有：
#   ix.chunks        所有片段（每個片段有 id / path / heading / text）
#   ix.bm25          關鍵字檢索用的 BM25 索引
#   ix.store         向量存放處（沒建向量或載入失敗時是 None）
#   ix.has_vectors   向量能不能用（True / False）
# embedding 參數：EMBEDDING 環境變數沒設就用 "local"（本機跑 e5-small 模型）；設 "none" 就完全不載向量。
ix = load_index(Path("data"), embedding=os.getenv("EMBEDDING", "local"))

# {c.path for c in ix.chunks} 是 set comprehension：把所有片段的檔案路徑收成集合，len 就是檔案數
print(f"{len(ix.chunks)} 個片段 / {len({c.path for c in ix.chunks})} 個檔案")
print(f"向量：{'可用' if ix.has_vectors else '不可用（降級為純 BM25，HyDE 與 MMR 效果會打折）'}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Building prefix dict from the default dictionary ...


Loading model from cache /var/folders/d3/00wx88zj1d17mtmk5gj09vpm0000gn/T/jieba.cache


Loading model cost 0.186 seconds.


Prefix dict has been built successfully.


41 個片段 / 10 個檔案
向量：可用


## Step 1 · 先看模組庫

七個模組，依 Modular RAG 的階段分類。**每個模組就是一個工具**。

In [2]:
from collections import defaultdict

# M.MODULES 是一個 dict：{模組名: {"stage": 階段, "fn": 函式, "description": 描述, "schema": 參數規格}}
# 這格把七個模組按 stage 分組印出來。
#
# defaultdict(list)：跟一般 dict 一樣，但第一次碰到不存在的 key 時會自動給一個空 list，
# 所以可以直接 by_stage[stage].append(...)，不用先檢查 key 在不在。
by_stage = defaultdict(list)
for name, mod in M.MODULES.items():
    by_stage[mod["stage"]].append(name)

# 依 Modular RAG 的四個階段順序印。Indexing 這階段目前只有 list_corpus 一個模組。
for stage in ["Indexing", "Pre-retrieval", "Retrieval", "Post-retrieval"]:
    print(f"\n【{stage}】")
    for name in by_stage[stage]:
        # 描述可能有好幾句，只印第一句：用「。」切開取第 0 段
        first_line = M.MODULES[name]["description"].split("。")[0]
        print(f"  {name:18} {first_line}。")     # {name:18} = 名稱欄固定 18 格寬，對齊用


【Indexing】
  list_corpus        列出知識庫有哪些檔案、各有幾個片段。

【Pre-retrieval】
  multi_search       RAG-Fusion：一次送多個不同角度的 query，結果用 RRF 融合。
  hyde_search        HyDE：先自己寫一段「答案大概長這樣」的假設文字，拿那段去做向量檢索。

【Retrieval】
  search             檢索知識庫。

【Post-retrieval】
  grade_documents    取回片段的完整內文，讓你評估夠不夠回答問題。
  expand             取某片段在原文中的前後鄰居。
  diversify          MMR 去重：從一堆候選片段裡挑出「相關且彼此不重複」的子集。


### 模組的原始碼隨時可以看

`module_source()` 用 `inspect.getsource()` 即時抓，所以你看到的永遠是正在跑的程式碼。

In [3]:
# 印出 hyde_search 這個模組的原始碼。
# 重點：模組就是普通的 Python 函式，第一個參數固定是 ix（索引），其餘是模型可以填的參數。
# 你會看到它只做「拿假答案去做向量檢索」這一件事 —— 沒有任何判斷邏輯，判斷都留給模型。
print(M.module_source("hyde_search"))

def hyde_search(ix: Index, hypothetical_answer: str, k: int = 5) -> dict:
    """[Pre-retrieval] HyDE：不要拿問題去比對，拿你「猜的答案」去比對。

    問題和答案的語意分佈不一樣 —— 問句短、抽象；文件是陳述句、具體。
    先請模型憑空寫一段「答案看起來會長這樣」，再拿那段去做向量檢索，
    比對的就是陳述句對陳述句，命中率通常明顯提升。

    注意這裡「假答案」可能整段是錯的 —— 沒關係，我們要的是它的用詞與句式。
    """
    hits = vector_search(ix, hypothetical_answer, k=k)
    return {
        "hypothetical_answer": hypothetical_answer[:200],
        "note": "用假設答案而非原問題做檢索（HyDE）",
        "results": _hits_to_results(ix, hits),
        "degraded": "向量不可用，退回 BM25（HyDE 在純 BM25 下效果會打折）"
        if hits and hits[0].degraded
        else None,
    }



## Step 2 · 架構 = 模組 + 編排

看 `Architecture` 這個 dataclass —— 整套教材的主張都在這裡：

In [4]:
import inspect
# inspect.getsource() 把一個 class 或函式的原始碼抓出來當字串。
# Architecture 是一個 dataclass（會自動幫你產生 __init__ 的 class），欄位分兩類：
#   name / paper / orchestration   純說明，給人看的
#   modules                        這個架構拿得到哪些工具（名稱清單）—— 決定它「能做什麼」
#   policy                         編排規則，會接在 BASE_POLICY 後面變成 system prompt —— 決定它「怎麼做」
#   builtin_tools                  要不要開 SDK 內建工具（例如 WebSearch），預設不開
#   max_turns                      最多讓 agent 跑幾輪，防止無限迴圈
print(inspect.getsource(M.Architecture))

@dataclass
class Architecture:
    """一個 RAG 架構 = 用哪些模組 + 怎麼編排它們。

    這個 dataclass 就是整套教材的核心主張：CRAG 和 Self-RAG 的差別不在程式碼，
    在 policy 那段字串裡。
    """

    name: str
    paper: str  # 出處
    orchestration: str  # Linear / Conditional / Branching / Looping
    modules: list[str]  # 用到哪些模組
    policy: str  # 這個架構的編排規則
    builtin_tools: list[str] = field(default_factory=list)  # SDK 內建工具，如 WebSearch
    max_turns: int = 12

    def system_prompt(self) -> str:
        return f"{BASE_POLICY}\n\n## 這次採用的流程：{self.name}\n\n{self.policy.strip()}"

    def summary(self) -> str:
        stages = sorted({MODULES[m]["stage"] for m in self.modules if m in MODULES})
        tools = self.modules + [f"{t}(SDK 內建)" for t in self.builtin_tools]
        return (
            f"{self.name}  [{self.orchestration}]\n"
            f"  出處：{self.paper}\n"
            f"  階段：{' → '.join(stages)}\n"
            f"  模組：{', '.join(tools)}"
        )



In [5]:
# M.ARCHITECTURES 是 {代號: Architecture 物件}，這本會跑的六個都在裡面，還有幾個練習用的。
# summary() 把一個架構的名稱 / 出處 / 階段 / 模組印成幾行。
# .replace("\n", "\n  ") 只是在每一行前面多加兩格縮排，純排版。
for key, arch in M.ARCHITECTURES.items():
    print(f"[{key}]")
    print("  " + arch.summary().replace("\n", "\n  "))
    print()

[naive]
  Naive RAG  [Linear]
    出處：Lewis et al. 2020（原始 RAG）
    階段：Retrieval
    模組：search

[rewrite]
  Rewrite-Retrieve-Read  [Linear（Pre-retrieval 改寫）]
    出處：Ma et al. 2023《Query Rewriting for Retrieval-Augmented LLMs》arXiv:2305.14283
    階段：Retrieval
    模組：search

[hyde]
  HyDE  [Linear（Pre-retrieval 轉換）]
    出處：Gao et al. 2022《Precise Zero-Shot Dense Retrieval without Relevance Labels》
    階段：Pre-retrieval → Retrieval
    模組：hyde_search, search

[rag_fusion]
  RAG-Fusion（多查詢）  [Branching]
    出處：Rackauckas 2024 / Query Expansion 系列
    階段：Pre-retrieval
    模組：multi_search

[self_ask]
  Self-Ask  [Linear（顯式問題分解）]
    出處：Press et al. 2022《Measuring and Narrowing the Compositionality Gap》arXiv:2210.03350
    階段：Retrieval
    模組：search

[ircot]
  IRCoT（交錯檢索與推理）  [Looping（推理與檢索交錯）]
    出處：Trivedi et al. 2023《Interleaving Retrieval with Chain-of-Thought Reasoning》arXiv:2212.10509
    階段：Post-retrieval → Retrieval
    模組：search, expand

[crag]
  CRAG（Corrective RAG）  [Conditional（三分支

## Step 3 · Naive RAG —— 先看它怎麼失敗

基準線。`modules=["search"]`、`Linear`、不准重試。

**注意它結構上的限制**：它手上根本沒有 `grade_documents`，
所以它**不可能**知道自己檢索失敗了 —— 這不是模型笨，是架構沒給它這個能力。

問一個刻意口語化的問題，文件裡根本沒有「擋東西」這三個字：

In [6]:
# 這一題整本 notebook 會反覆用，所以存成常數。
# 故意講得很口語：文件裡寫的是 hook / permissions，沒有「擋東西」這三個字，
# 純關鍵字比對會很吃力，才看得出各架構的差別。
QUESTION = "那個會擋東西的功能怎麼設？"

print(M.NAIVE.policy)      # 先看 Naive 的編排規則長什麼樣（就一句話：查一次就答）
print("─" * 60)

# M.arun(架構, 問題, 索引) 是跑一次 agent 的入口。它是 async 函式，所以要 await。
# 跑的時候會即時印出軌跡：
#   [1] search(...)   第幾步呼叫了什麼工具、參數是什麼
#        → 8 筆       工具回傳的摘要
#   · 開頭的那行      模型呼叫工具前寫的理由（BASE_POLICY 要求它每次都要講）
#
# 回傳一個 Run 物件，之後比較架構全靠它：
#   .answer        最後的答案（字串）
#   .tool_calls    每次工具呼叫的紀錄（list）
#   .n_calls       呼叫次數
#   .citations     答案裡引用到的片段 id（像 [02-permissions.md#0]）
#   .elapsed_s     花了幾秒
#   .trace()       把工具呼叫排成一段文字
naive_run = await M.arun(M.NAIVE, QUESTION, ix)
print("\n【答案】\n" + naive_run.answer[:900])   # [:900] 只印前 900 字，答案太長會刷版

檢索一次，然後根據結果作答。

不要重試、不要改寫 query、不要評估品質 —— 就查一次。
這是基準線，用來對照後面幾種架構好在哪。
────────────────────────────────────────────────────────────


  [1] search({"query": "擋住 阻擋 封鎖 攔截 禁止 權限設定", "method": "hybrid", "k": 8})
       → 8 筆


  · 使用者用的是模糊口語（「會擋東西」），沒有明確術語可以精確匹配，這種情況 bm25 容易失準，但也不排除文件裡真有「阻擋/封鎖」字樣，所以用 hybrid 兼顧兩邊，一次查完。



【答案】
你說的「擋東西」，知識庫裡其實有**兩套獨立機制**，都寫在同一個 `settings.json`，但互不干涉——搞混這兩個是最常見的誤解 [02-permissions.md#0]。

## 差別在哪

| | Hook | Permission |
|---|---|---|
| 判斷方式 | 執行腳本看 exit code | 比對規則字串 |
| 複雜判斷 | 可以（任意程式邏輯） | 不行（只有模式比對） |
| 速度 | 要開行程，較慢 | 極快 |
| 會不會問使用者 | 不會，直接擋 | 可以設成詢問 |

[02-permissions.md#0]

## 一、Permission：規則比對，設定最快

分 `allow` / `deny` / `ask` 三個區塊 [02-permissions.md#1]：

```json
{
  "permissions": {
    "allow": [
      "Bash(git status)",
      "Bash(git diff:*)",
      "Read(**)"
    ],
    "deny": [
      "Bash(rm -rf:*)",
      "Read(.env)",
      "Write(/Users/**/.ssh/**)"
    ],
    "ask": [
      ...
    ]
  }
}
```

另外整個 session 還有模式可切 [02-permissions.md#2]：`default`（照規則走，沒規則就問）、`acceptEdits`（檔案編輯自動同意）、`plan`（唯讀，不能改任何東西）、`bypassPermissions`（全部放行，只在沙箱或工具已受限時用）。

⚠️ 常見誤解：Agent SDK 裡 `allowed_tools` **不等於**「只能用這些工具」，它只是「不用問就能用」的白名單，沒列出的工具仍然存在、只是會觸發詢問。要真正關掉工具得用 `tools` 參數 [02-permission


## Step 4 · CRAG —— 加一個評估模組，行為就變了

`CRAG`（Yan et al. 2024）比 Naive 多兩件事：

1. 多一個模組：`grade_documents`（evaluator）
2. 編排從 Linear 變成 **Conditional 三分支**：CORRECT / AMBIGUOUS / INCORRECT

程式碼一行都沒改，只是換一份 `Architecture`：

In [7]:
# 先不跑，只看設定：CRAG 跟 Naive 差在 modules 多了 grade_documents、policy 多了三分支的規則。
print(M.CRAG.summary())
print("─" * 60)
print(M.CRAG.policy)

CRAG（Corrective RAG）  [Conditional（三分支 + web fallback）]
  出處：Yan et al. 2024《Corrective Retrieval Augmented Generation》
  階段：Post-retrieval → Retrieval
  模組：search, grade_documents, expand, WebSearch(SDK 內建)
────────────────────────────────────────────────────────────
照 CRAG 的三分支流程走：

1. 先用 search 檢索**知識庫**。
2. 用 grade_documents 取回完整內文，逐塊給 0-2 分。
3. 依整體評分走分支：
   - **CORRECT**（多數 2 分）→ 直接作答。
   - **AMBIGUOUS**（混雜 1 分）→ 改寫 query 再檢索一輪，把兩輪結果合併後作答。
     必要時用 expand 補上下文。
   - **INCORRECT**（多數 0 分）→ 知識庫裡沒有這件事。這時**才**用 WebSearch 上網找。

**WebSearch 的三條規矩**（很重要，這是可信度的分水嶺）：

- 只在走到 INCORRECT 分支時用。知識庫有答案就不要上網 —— 內部文件才是這個系統的權威來源。
- 網路來源和知識庫來源**必須分開標**：
  知識庫用 `[CLAUDE.md#12]`，網路用 `[web: 網站名 - 完整網址]`。
- 答案開頭要有一句話說明「這題知識庫沒有，以下來自網路」，不要混在一起讓使用者分不出來。

每次分支判斷都要說出你選了哪一支、為什麼。


In [8]:
# 同一題、同一個 arun()，只是把 M.NAIVE 換成 M.CRAG。
# 觀察軌跡：應該會看到 search → grade_documents，評分不理想的話再 search 一次。
crag_run = await M.arun(M.CRAG, QUESTION, ix)
print("\n【答案】\n" + crag_run.answer[:1200])

  [1] search({"query": "阻擋 攔截 禁止 設定", "method": "hybrid", "k": 6})
       → 6 筆


  [2] search({"query": "deny 權限 拒絕 允許清單 permissions", "method": "hybrid", "k": 6})
       → 6 筆


  [3] search({"query": "hook 攔截工具呼叫 block", "method": "hybrid", "k": 6})
       → 6 筆


  · 問題描述很模糊（「擋東西」可能指權限拒絕、hook 攔截、或過濾規則），這種使用者自己的話用語意檢索比較容易對上，我先用 hybrid 撒幾個不同角度的網。


  [4] grade_documents({"question": "怎麼設定「會擋東西」的功能（阻擋危險操作）", "chunk_ids": ["02-permissions.md#0", "02-permissions.md#1", "02-permissi)
       → 取回 7 塊完整內文


  · 知識庫命中很集中：有「Permissions」和「Hooks」兩套阻擋機制。我取回完整內文逐塊評分，確認能不能直接作答。



【答案】
**評分結果：7 塊中 6 塊給 2 分**（02-permissions.md#0/#1/#2、01-hooks.md#3/#4、10-troubleshooting.md#0 都直接回答了設定方式），只有 02-permissions.md#3 給 1 分（講 SDK 層面的工具關閉，相關但偏題）。→ **CORRECT 分支，直接作答，不上網**。

另外提一下：你說的「擋東西」在知識庫裡剛好是原文用語——02-permissions.md 的標題就叫「Permissions：另一套『擋東西』的機制」，所以這題其實有**兩套**機制，先幫你分清楚要用哪一套。

---

## 先選：Permission 還是 Hook？

知識庫明講這是新手最常見的誤解——兩套是獨立機制，寫在同一個 `settings.json` 裡，互不干涉 [02-permissions.md#0]。

| | Hook | Permission |
|---|---|---|
| 判斷方式 | 執行腳本看 exit code | 比對規則字串 |
| 複雜判斷 | 可以（任意程式邏輯） | 不行（只有模式比對） |
| 速度 | 要開行程，較慢 | 極快 |
| 會問使用者嗎 | 不會，直接擋 | 可以設成詢問 |

判準一句話：**能寫成規則字串的用 permission，需要看內容才能決定的用 hook** [02-permissions.md#0]。

---

## 作法 A：Permissions（大多數情況用這個就夠）

`settings.json`：

```json
{
  "permissions": {
    "allow": [
      "Bash(git status)",
      "Bash(git diff:*)",
      "Read(**)"
    ],
    "deny": [
      "Bash(rm -rf:*)",
      "Read(.env)",
      "Write(/Users/**/.ssh/**)"
    ],
    "ask": [
      "Bash(git push:*)"
    ]
  }
}
```

- **allow** — 不用問就能執行
- **deny** — 

### 對照一下

看軌跡：CRAG 會出現 `grade_documents`，而且評分不理想時會**再 search 一輪**。
Naive 不會，因為它沒有那個工具。

> **教學金句**：「CRAG 不是更聰明的模型，是多給了它一個『可以承認失敗』的工具。」

In [9]:
# .trace() 把一次執行的工具呼叫排成清單：[步驟] 工具(參數) → 結果摘要
print(naive_run.trace())
print()
print(crag_run.trace())
print()
# M.compare([多個 Run]) 印一張表：架構 / 工具呼叫次數 / 耗時 / 出處數 / 答案長度
print(M.compare([naive_run, crag_run]))

── Naive RAG · 1 次呼叫 · 27.9s ──
  [1] search({"query": "擋住 阻擋 封鎖 攔截 禁止 權限設定", "method": "hybrid", "k": 8})
       → 8 筆

── CRAG（Corrective RAG） · 4 次呼叫 · 43.4s ──
  [1] search({"query": "阻擋 攔截 禁止 設定", "method": "hybrid", "k": 6})
       → 6 筆
  [2] search({"query": "deny 權限 拒絕 允許清單 permissions", "method": "hybrid", "k": 6})
       → 6 筆
  [3] search({"query": "hook 攔截工具呼叫 block", "method": "hybrid", "k": 6})
       → 6 筆
  [4] grade_documents({"question": "怎麼設定「會擋東西」的功能（阻擋危險操作）", "chunk_ids": ["02-permissions.md#0", "02-permissions.md#1", "0)
       → 取回 7 塊完整內文

架構                    工具呼叫  耗時     出處數  答案長度  
────────────────────  ────  ─────  ───  ──────
Naive RAG             1     27.9s  8    1941 字
CRAG（Corrective RAG）  4     43.4s  7    2588 字


### CRAG 的第三條分支：知識庫沒有的時候

論文的 CRAG 有三個分支，前面只示範了兩個：

| 分支 | 條件 | 動作 |
|---|---|---|
| CORRECT | 多數 2 分 | 直接作答 |
| AMBIGUOUS | 混雜 1 分 | 改寫 query 再查一輪 |
| **INCORRECT** | **多數 0 分** | **知識庫沒有 -> 上網找** |

第三條分支需要知識庫以外的來源。Claude Agent SDK 內建了 `WebSearch`，
所以**啟用它只要在架構上宣告一行**：

```python
CRAG = Architecture(
    ...,
    builtin_tools=["WebSearch"],   # <- 就這樣
)
```

`build_options()` 會把它交給 SDK：

```python
ClaudeAgentOptions(
    tools=arch.builtin_tools,                      # 只開架構明確要的內建工具，其餘全關
    allowed_tools=[...模組...] + arch.builtin_tools,
)
```

這裡有個可信度的分水嶺：一旦答案可能來自網路，使用者就必須分得出哪句來自你的文件、
哪句來自網路。所以 CRAG 的 policy 多了三條規矩：

1. 只在 INCORRECT 分支才用 —— 知識庫有答案就不要上網
2. 出處分開標：知識庫 `[CLAUDE.md#12]`，網路 `[web: 網站名 - 網址]`
3. 答案開頭要明講「這題知識庫沒有，以下來自網路」

下面問一個**知識庫絕對沒有**的問題，看它走哪條路：

In [10]:
# 問一題知識庫絕對沒有的（體育新聞），逼 CRAG 走 INCORRECT 分支。
# 預期軌跡：search → grade_documents（全部 0 分）→ WebSearch（SDK 內建工具，不是我們寫的）。
# 注意 WebSearch 會真的連網，比較慢；沒網路的話這格會失敗。
web_run = await M.arun(M.CRAG, "2026 年溫布頓網球男單冠軍是誰？", ix)
print("\n【答案】\n" + web_run.answer[:1100])

  [1] search({"query": "2026 溫布頓 男單 冠軍", "method": "hybrid", "k": 5})
       → 5 筆


  [2] search({"query": "Wimbledon tennis champion", "method": "bm25", "k": 5})
       → 0 筆


  · 這題雖然看起來不像技術知識庫的內容，但 CRAG 流程規定先查知識庫再說，我用 hybrid 兼顧關鍵字與語意。


  [3] grade_documents({"question": "2026 年溫布頓網球男單冠軍是誰？", "chunk_ids": ["09-rag-patterns.md#1", "09-rag-patterns.md#4", "07-agent-sdk)
       → 取回 3 塊完整內文


  · BM25 一個詞都沒命中，hybrid 回的全是 RAG/Agent SDK 技術文件。我還是照流程取回完整內文確認評分，避免只看摘要就誤判。


...

  [5] WebSearch({"query": "溫布頓 2026 男單 冠軍"})


       → 2 筆


       → 2 筆


  · 評分結果：三塊全部 **0 分**（都是 Agentic RAG 教材與 Agent SDK 認證設定，跟網球毫無關係）。這是典型的 **INCORRECT 分支** —— 知識庫是技術文件語料，根本不含體育賽事資料。依規矩，此時才啟用 WebSearch。



【答案】
**這題知識庫沒有，以下答案來自網路。**

知識庫（Agentic RAG / Claude Agent SDK 教材）完全不含體育賽事資料，檢索到的片段全部評 0 分，屬 CRAG 的 INCORRECT 分支，因此才轉用 WebSearch。

## 結論

**2026 年溫布頓男單冠軍：揚尼克·辛納（Jannik Sinner，義大利）**

- 決賽擊敗 **亞歷山大·茲韋列夫（Alexander Zverev）**，比分 **6–7(7–9)、7–6(7–2)、6–3、6–4** [web: Olympics.com - https://www.olympics.com/en/news/wimbledon-2026-jannik-sinner-alexander-zverev-mens-singles-final-resuls]
- 這是他**衛冕**溫網（2025、2026 連兩年），個人**第 2 座溫網**、**第 5 座大滿貫**單打冠軍 [web: Wikipedia - https://en.wikipedia.org/wiki/2026_Wimbledon_Championships_%E2%80%93_Men%27s_singles]
- 官方籤表確認：THE CHAMPIONSHIPS 2026 GENTLEMEN'S SINGLES Champion: SINNER, Jannik ITA [web: Wimbledon 官網 - https://www.wimbledon.com/en_GB/scores/draws/2026_MS_draw.pdf]

> ⚠️ 一點提醒：中文搜尋結果中有一則自由體育的標題寫「逆轉二連霸艾卡拉茲」，與其他來源（決賽對手為茲韋列夫）不一致，可能是舊聞或標題混用。以 Wimbledon 官方籤表與 Olympics.com 的賽果為準。

## 檢索過程摘要

| 步驟 | 判斷 |
|---|---|
| search（hybrid + bm25） | bm25 零命中；hybrid 只回 RAG 架構表、RRF 公式、SDK 認證 |
| grade_documents | `09-rag-patterns.md#1` = 0、`09-rag-patterns.md#4` = 0、`

### 這段軌跡要停下來講三件事

**一、它沒有直接上網。** 先查了知識庫、評分確認全是 0 分，**才**走 INCORRECT 分支。
policy 寫「只在 INCORRECT 分支用」它就真的忍住了 —— 這是可驗證的行為，不是你希望它這樣。

**二、平行呼叫。** 它常常一次發兩個 WebSearch（中英文各一）。
軌跡用 `tool_use_id` 對應步驟而不是用計數器，所以結果亂序回來也不會疊錯卡片。

**三、出處真的分開了。** 網路來源長 `[web: Olympics.com - https://...]`，
跟知識庫的 `[CLAUDE.md#12]` 一眼分得出來。

> **教學金句**：「RAG 加上網搜，等於把幻覺的邊界往外推。你至少要讓使用者看得出這句話是從哪裡來的。」

## Step 5 · HyDE —— 在檢索「之前」動手腳

前面兩個都在檢索**之後**想辦法。HyDE（Gao et al. 2022）反過來，在檢索**之前**改造查詢。

核心觀察：**問題和答案的語意分佈不一樣。** 問句短、抽象、疑問式；文件是長的、具體的陳述句。
拿問句去比對陳述句，本來就吃虧。

HyDE 的作法：先讓模型憑空寫一段「答案大概長這樣」，拿**那段假答案**去做向量檢索。

假答案的內容可能整段是錯的 —— 不重要，我們要的是它的**用詞和句式**。

In [11]:
# HyDE：問題也刻意口語。「一堆底線」指的是 mcp__server__tool 這種工具命名。
# 觀察軌跡：模型會先自己寫一段「假答案」，然後呼叫 hyde_search(hypothetical_answer=那段假答案)，
# 工具拿假答案去做向量檢索。
hyde_run = await M.arun(M.HYDE, "工具名字前面那一堆底線是幹嘛的？", ix)
print("\n【答案】\n" + hyde_run.answer[:900])

  [1] hyde_search({"hypothetical_answer": "MCP 伺服器提供的工具在註冊時會加上前綴，命名格式為 mcp__<伺服器名稱>__<工具名稱>，中間以雙底線分隔。這樣做是為了避免不同 MCP 伺服器之間、以及與內建工)
       → 5 筆


  [2] search({"query": "mcp__ 工具名稱 前綴 雙底線 命名", "method": "bm25", "k": 5})
       → 5 筆


  · 這題問的是工具命名慣例（`mcp__ragmod__hyde_search` 那串底線），我先用 HyDE 寫一段「文件大概會這樣解釋命名規則」的假設文字去檢索，這種格式描述通常語意句式比關鍵字好對上；同時併一發 bm25 直接打 `mcp__` 這個字串，因為它是精確術語。



【答案】
那是 **MCP 工具的命名前綴**。

## 格式

```
mcp__<server 名>__<工具名>
```

MCP 工具在 Claude 眼中的名字就是這個樣子，中間用雙底線分隔 [06-mcp.md#1]。例如一個叫 `sqlite` 的 MCP server 提供 `query` 工具，完整名稱就是 `mcp__sqlite__query` [06-mcp.md#1]。

同理，你現在看到的 `mcp__ragmod__hyde_search`，就是 `ragmod` 這個 server 提供的 `hyde_search` 工具。

## 實務上要注意的踩雷點

寫 `allowed_tools` 的時候**必須用完整名稱**，這是很常見的錯誤 [06-mcp.md#1]。排錯文件也特別列了這一條：「`allowed_tools` 寫了工具名還是被擋 → MCP 工具的完整名稱是 `mcp__<server>__<tool>`，要用完整名稱」[10-troubleshooting.md#1]。

實際寫法像這樣：

```python
ClaudeAgentOptions(
    tools=[],                      # 關閉所有內建工具
    allowed_tools=["mcp__x__y"],   # 這些不用問
)
```
[02-permissions.md#3]

順帶一提，`tools` 只管內建工具，MCP 工具走 `mcp_servers` 參數，兩者獨立；所以設 `tools=[]` 不會讓 MCP 工具消失 [10-troubleshooting.md#1]。

補充一點：知識庫裡有提到前綴機制，但**沒有明講「為什麼」要加前綴**（例如是否為了避免命名衝突），這部分查不到，我就不臆測了。


⚠️ **HyDE 需要向量檢索才有意義**。如果上面顯示「向量不可用」，它會退回 BM25，
那就等於拿一段長文去做關鍵字比對 —— 效果會很怪。這是教學時要講清楚的前提條件。

## Step 6 · RAG-Fusion —— Branching 編排

不是查一次，是**同時查很多個角度**再用 RRF 融合。

為什麼有效：單一 query 的用詞偏好決定了命中什麼。丟 3-5 個不同問法，
同時出現在多份排行榜上的片段自然浮到前面 —— 這比賭一次查詢的用詞穩得多。

RRF 的公式只有一行：`score = Σ 1/(60 + rank)`。**零參數**，
所以不用煩惱 BM25 分數（0～40）和 cosine（0～1）怎麼加權。

In [12]:
# RAG-Fusion：同一題 QUESTION。
# 觀察軌跡：模型會呼叫 multi_search(queries=[3～5 個不同問法])，
# 工具內部把每個問法各查一次，用 RRF 融合成一份排名再回傳。
fusion_run = await M.arun(M.RAG_FUSION, QUESTION, ix)
print("\n【答案】\n" + fusion_run.answer[:900])

  [1] multi_search({"queries": ["如何設定權限 deny 拒絕規則 permissions", "hooks 攔截阻止工具執行 PreToolUse block", "封鎖禁止指令 blocklist 不允許執行的命令", ")
       → 8 筆


  · 問題很模糊（「會擋東西的功能」），可能指權限拒絕清單、hooks 攔截、或工具封鎖，所以我用多角度改寫一次撈，讓同時命中多個榜的片段浮上來；語意+關鍵字都需要，用 hybrid。



【答案】
知識庫裡「擋東西」其實有**兩套機制**，用途不同：

## 1. Permissions（設定檔宣告式，最直接）

在 `.claude/settings.json` 用 allow / deny / ask 三個區塊 [02-permissions.md#1]：

```json
{
  "permissions": {
    "allow": [
      "Bash(git status)",
      "Bash(git diff:*)",
      "Read(**)"
    ],
    "deny": [
      "Bash(rm -rf:*)",
      "Read(.env)",
      "Write(/Users/**/.ssh/**)"
    ],
    "ask": [
      "Ba..."
    ]
  }
}
```

**最容易踩的坑**：在 Agent SDK 裡 `allowed_tools` 不等於「只能用這些工具」，它只是「不用問就能用」的白名單，沒列到的工具仍然存在、只是會跳詢問。要真正把工具**關掉**要用 `tools` 參數 [02-permissions.md#3]：

```python
ClaudeAgentOptions(
    tools=[],                      # 關閉所有內建工具
    allowed_tools=["mcp__x__y"],   # 這些不用問
)
```

補充：`tools` 只管內建工具，MCP 工具走 `mcp_servers`，兩者獨立；MCP 工具名要寫完整的 `mcp__<server>__<tool>`，不然會一直被擋 [10-troubleshooting.md#1]。

## 2. Hooks（用程式碼焊死，能擋得更細）

`PreToolUse` 在工具真正執行**之前**觸發，典型用途就是阻止危險操作 [01-hooks.md#1]。腳本光放在資料夾不會生效，必須註冊進 `.claude/settings


## Step 7 · Self-RAG —— Looping 編排

Self-RAG（Asai et al. 2024）的原作是**訓練**模型輸出四種反思 token
（`Retrieve?` / `IsRel` / `IsSup` / `IsUse`）。

我們沒有要訓練模型，所以改成**把那四個問題寫進 policy，要求它每輪明講**：

| 論文的 reflection token | 我們的作法 |
|---|---|
| `Retrieve?` 要不要檢索 | 「這題如果知識庫沒有就答不了嗎？」 |
| `IsRel` 檢索到的相關嗎 | 用 `grade_documents` 判斷 |
| `IsSup` 答案有被支撐嗎 | 「逐句檢查，沒出處的刪掉」 |
| `IsUse` 對使用者有用嗎 | 「不夠完整就再補一輪」 |

**這個取捨要在課堂上講清楚**：訓練版本更省 token、更穩定；
prompt 版本零訓練成本、可以當場改 —— 但它會把反思寫成文字，比較囉嗦也比較貴。

In [13]:
# Self-RAG 用一題「需要比較兩個概念」的問題，才有東西可以反思。
# 觀察軌跡：檢索 → 評分 → 寫草稿 → 自我批判「哪句沒出處」→ 不夠就再查。
# 這是 Looping 編排，會比前面幾個慢很多（常常 60～90 秒），正常。
self_rag_run = await M.arun(M.SELF_RAG, "sub-agent 跟 agent team 差在哪？什麼時候該用哪個？", ix)
print("\n【答案】\n" + self_rag_run.answer[:1200])

  [1] search({"query": "sub-agent agent team", "method": "bm25", "k": 6})
       → 6 筆


  [2] search({"query": "什麼時候該用子代理，什麼時候該用多代理團隊，兩者的差別", "method": "vector", "k": 6})
       → 6 筆


  · ## 第 1 輪反思

**需要檢索嗎？** 需要。「sub-agent」和「agent team」是知識庫裡的特定術語，不同專案對這兩個詞的定義差很多，不能靠背景知識推測。

這兩個是明確的術語名稱，關鍵字比對會比語意相似更準；但使用者問的是「差在哪、何時用哪個」這種比較性問題，語意檢索可能撈到比


  [3] grade_documents({"question": "sub-agent 跟 agent team 的差別，以及各自適用時機", "chunk_ids": ["03-subagents.md#0", "03-subagents.md#1", "0)
       → 取回 4 塊完整內文


  · 兩個查詢高度收斂到同一份文件 `03-subagents.md` 的三個連續片段，這是好訊號（不是各說各話）。但 snippet 有被截斷（結尾是「…」），關鍵的對照表和 context 說明沒看完，得取完整內文才能判斷夠不夠答。順便把 skills 那份的分工表一起評，它可能補充 sub-agen


  [4] expand({"chunk_id": "03-subagents.md#2", "window": 1})
       → 取回 3 塊完整內文


  · ## 第 2 輪反思

**檢索到的相關嗎？** 評分如下：

| 片段 | 分數 | 理由 |
|---|---|---|
| `03-subagents.md#0` | **2** | Sub-agent 定義、省 context 的機制、定義檔寫法、不能互通的限制 |
| `03-subage



【答案】
## 第 3 輪反思

**草稿有被支撐嗎？** 已確認 `#2` 是該文件最後一段（下一塊已跳到 `04-skills.md`），所以 git worktree 在知識庫裡只有「通常配合」這一句、沒有更多細節——我不會自己補。以下每句都對得上片段。

**這樣夠有用嗎？** 夠：定義、對照、時機、踩坑點都齊了。

---

# 結論

一句話：**sub-agent 是「派出去拿結論」，agent team 是「一群人一起施工」**。差別的核心在**彼此能不能溝通**。

## 兩者的差別

| | Sub-agent | Agent team |
|---|---|---|
| 誰生的 | 主對話 | team lead |
| 彼此能溝通 | 不行 | 可以 |
| 有沒有獨立工作區 | 沒有 | 通常配合 git worktree |
| 適合 | 一次性的委派任務 | 需要協調的平行開發 |
| 成本 | 較低 | 較高 |

（表格出自 [03-subagents.md#1]）

**Sub-agent** 是主 Claude 用 Task 工具生出來的子代理，有自己的上下文視窗，做完只把**結論**回報主對話 [03-subagents.md#0]：

```
主對話
  ├─ Task → sub-agent A（自己的 context）→ 回報結論
  └─ Task → sub-agent B（自己的 context）→ 回報結論
```

它最大的價值是**省 context**：叫 sub-agent 去讀 30 個檔案找某個函式，那 30 個檔案不會佔用主對話視窗，主對話只收到一句「在 `retrieval.py:88`」[03-subagents.md#0]。代價是**sub-agent 之間不能互相講話**，各自獨立、只跟主對話溝通 [03-subagents.md#0]。

**Agent team** 是 team lead 生出來的獨立 Claude Code 實例，共享一張任務清單和一個信箱，**可以直接互傳訊息、對齊契約** [03-subagents.md#1]。

## 什麼時候用哪個

**用 sub-agent**：任務可以獨立完成、只需要結論、想省主對話的 context。例如「找出所有呼叫這個函式的地方

## Step 8 · Adaptive-RAG —— 依複雜度路由

前面每個架構都對所有問題一視同仁。但「今天星期幾」和「比較三份文件的差異」
需要的力氣差很多 —— 簡單問題跑複雜流程是浪費成本和延遲。

Adaptive-RAG（Jeong et al. 2024）先分類，再決定路線：

| 類別 | 什麼問題 | 走哪條 |
|---|---|---|
| A | 不需檢索（純語言任務、超出範圍） | 直接答 |
| B | 單一事實 | 檢索一次 |
| C | 需比較 / 多跳 / 綜合 | 多次檢索 + 驗證 |

⚠️ **跟論文的差異要講**：原論文用一個**訓練過的小分類器**做路由，延遲與成本都更低。
我們用 agent 自己判斷 —— 不用訓練，但分類本身也要花一次推論。

下面連跑三題，看它自己選了哪條路：

In [14]:
# 連跑三題難度不同的問題，看 Adaptive 有沒有替每一題選對路線。
# 每題的 Run 都收進 adaptive_runs，下一格用 compare 並排比。
adaptive_runs = []
for q in [
    "把「hook 會在工具執行前攔截」翻譯成英文",          # 預期 A：不需檢索，直接答（0 次工具呼叫）
    "PreToolUse hook 要 exit 多少才會擋住？",            # 預期 B：單一事實，查一次就好
    "hook 和 permissions 的差別是什麼？各自什麼時候用？",   # 預期 C：要比較兩個東西，多查幾次 + 驗證
]:
    print(f"\n{'=' * 70}\nQ: {q}\n{'=' * 70}")   # 分隔線 + 題目
    r = await M.arun(M.ADAPTIVE, q, ix)
    adaptive_runs.append(r)
    print("\n【答案】\n" + r.answer[:700])


Q: 把「hook 會在工具執行前攔截」翻譯成英文



【答案】
**分類：A 類｜不需檢索**

理由：這是純語言任務（中翻英），不涉及任何知識庫事實查詢。Adaptive-RAG 的核心取捨就在這裡——若無腦套用多步檢索流程，會白白付出檢索與驗證的延遲和 token 成本，而結果不會更好。（附帶一提，論文是用訓練好的小分類器做這個判斷，成本極低；這裡改由我自己推論分類，好處是免訓練、壞處是分類本身也要燒一次推論，對這種一眼可判的題目其實是多餘的開銷。）

翻譯結果：

> **The hook intercepts before the tool executes.**

其他語境下的變體：

- 偏技術文件語氣：*Hooks intercept tool calls before execution.*
- 強調時機點：*The hook fires before the tool runs, intercepting the call.*
- 若 hook 為專有名詞不譯：*The Hook will intercept prior to tool execution.*

（本題未使用知識庫，故無引用出處。）

Q: PreToolUse hook 要 exit 多少才會擋住？


  [1] search({"query": "PreToolUse hook exit code 2 block", "method": "bm25"})
       → 5 筆


  · **分類：B 類｜單步檢索**

理由：這是單一事實查詢（一個特定 exit code 的數值），不需要比較或多跳推理，一次檢索就能定案。用 Adaptive-RAG 的好處是省下多輪檢索的成本；代價是「分類」這步本身也花了一次推論——對這種一眼就能判定的簡單題，其實有點多餘。

「PreToolU



【答案】
# 結論：`exit 2`

PreToolUse hook 要 **exit code 為 `2`** 才會阻止工具執行 [10-troubleshooting.md#0]。

完整的 exit code 對照表 [01-hooks.md#2]：

| exit code | 意義 | Claude 會怎樣 |
|---|---|---|
| `0` | 通過 | 照常執行工具 |
| `2` | **阻止** | 工具不會執行，stderr 的內容回傳給 Claude 當理由 |
| 其他非零 | 錯誤 | 記錄錯誤，但不阻止 |

兩個重點：

1. **`exit 1` 不會擋**，只會記錄錯誤，工具照樣跑 [10-troubleshooting.md#0]。
2. `exit 2` 不只是擋下來，還會把你寫在 **stderr 的訊息餵回給模型**，所以模型看得到「為什麼被擋」[01-hooks.md#2]。

所以腳本裡要這樣寫：

```bash
echo "不可以寫入 .env" >&2
exit 2
```

**順帶一提**，如果 hook 根本沒反應（不是 exit code 的問題），最常見的兩個原因是：只把腳本丟進 `.claude/hooks/` 卻沒在 `settings.json` 的 `hooks` 欄位註冊、以及改了設定沒重開 session；另外別忘了 `chmod +x` [10-troubleshooting.md#0]。

Q: hook 和 permissions 的差別是什麼？各自什麼時候用？


...

  [2] multi_search({"queries": ["hooks 是什麼 如何設定", "permissions 權限設定 allow deny", "hook 和 permission 的差別", "什麼時候該用 hook 而不是 permis)
       → 8 筆


  · **分類：C 類｜多步檢索**

理由：這題要「比較兩個機制的差異」＋「各自適用時機」，是兩個子問題、兩個來源主題，單次檢索很容易只撈到其中一邊。用 multi_search 從多角度打，再驗證。（附帶說明取捨：論文是用訓練好的小分類器做這步，延遲/成本更低；我這裡靠自己判斷，省了訓練成本，但分類本


  [3] grade_documents({"question": "hook 和 permissions 的差別，各自什麼時候用", "chunk_ids": ["02-permissions.md#0", "02-permissions.md#1", "02)
       → 取回 6 塊完整內文


  · 命中了一個直接叫「Hook 與 Permission 的差別」的片段，我把 permissions 全檔和 hooks 開頭抓完整內文來驗證：



【答案】
**驗證結果：6 份中 5 份 2 分、1 份 1 分 → 資料充分，直接作答。**

## 一句話結論

> **能寫成規則字串的用 permission，需要看內容才能決定的用 hook。** [02-permissions.md#0]

這是 Claude Code 裡**兩套獨立**的阻擋機制，寫在同一個 `settings.json` 但互不干涉 —— 搞混是新手最常見的誤解 [02-permissions.md#0]。

## 差別對照

| | Hook | Permission |
|---|---|---|
| 判斷方式 | 執行一支腳本，看 exit code | 比對規則字串 |
| 能不能做複雜判斷 | 可以（任意程式邏輯） | 不行（只有模式比對） |
| 速度 | 要開行程，較慢 | 極快 |
| 會不會問使用者 | 不會，直接擋 | 可以設成詢問 |
| 適合什麼 | 需要看內容才能決定的規則 | 一眼就能判斷的黑白名單 |

[02-permissions.md#0]

本質差異：Hook 是「特定事件發生時自動執行的 shell 指令」，它不是給 Claude 的建議，而是**管 Claude 的 harness**，Claude 沒辦法忽略它 [01-hooks.md#0]。

## 各自什麼時候用

**用 permission 的場合**：黑白名單類、一眼能判斷的規則 [02-permissions.md#0]。三個區塊 —— `allow` 不用問就執行、`deny` 直接拒絕連問都不問、`ask` 每次都問 [02-permissio


In [15]:
# 三題並排。看「工具呼叫」那一欄：A 題應該是 0，B 題 1～2，C 題更多。
print(M.compare(adaptive_runs))
print("\n工具呼叫次數應該隨複雜度遞增 —— 這就是 Adaptive 的價值：簡單問題不要花大錢。")

架構            工具呼叫  耗時     出處數  答案長度  
────────────  ────  ─────  ───  ──────
Adaptive-RAG  0     11.2s  0    484 字 
Adaptive-RAG  1     16.8s  2    642 字 
Adaptive-RAG  3     35.9s  8    1748 字

工具呼叫次數應該隨複雜度遞增 —— 這就是 Adaptive 的價值：簡單問題不要花大錢。


## Step 9 · 並排比較

同一題、不同架構。這張表是課堂上最有說服力的一頁。

In [16]:
# 把前面跑過的三個 Run 拿來並排。它們問的都是同一題 QUESTION，所以可以比。
print(f"題目：{QUESTION}\n")
print(M.compare([naive_run, fusion_run, crag_run]))

# .citations 是從答案裡用正規表達式撈出來的 [檔名#編號] 清單。
# 清單是空的 = 答案沒有一句引用知識庫 = 很可能是模型憑記憶編的。
print("\n\n── 出處 ──")
for r in [naive_run, fusion_run, crag_run]:
    print(f"{r.architecture:22} {r.citations}")   # {…:22} 架構名欄固定 22 格寬

題目：那個會擋東西的功能怎麼設？

架構                    工具呼叫  耗時     出處數  答案長度  
────────────────────  ────  ─────  ───  ──────
Naive RAG             1     27.9s  8    1941 字
RAG-Fusion（多查詢）       1     28.5s  8    1807 字
CRAG（Corrective RAG）  4     43.4s  7    2588 字


── 出處 ──
Naive RAG              ['01-hooks.md#2', '01-hooks.md#4', '01-hooks.md#5', '02-permissions.md#0', '02-permissions.md#1', '02-permissions.md#2', '02-permissions.md#3', '10-troubleshooting.md#0']
RAG-Fusion（多查詢）        ['01-hooks.md#1', '01-hooks.md#4', '01-hooks.md#5', '02-permissions.md#1', '02-permissions.md#3', '07-agent-sdk.md#5', '10-troubleshooting.md#0', '10-troubleshooting.md#1']
CRAG（Corrective RAG）   ['01-hooks.md#3', '01-hooks.md#4', '02-permissions.md#0', '02-permissions.md#1', '02-permissions.md#2', '02-permissions.md#3', '10-troubleshooting.md#0']


### 怎麼讀這張表

- **工具呼叫次數**：不是越多越好。多代表它在自我修正，也代表更慢更貴。
- **出處數**：0 個出處 = 它在憑記憶答，這是 RAG 專案最常見的假成功。
- **耗時**：CRAG / Self-RAG 常常是 Naive 的 2-4 倍。你的產品能不能接受？

> **沒有最好的架構，只有適合這個問題的架構。** 這就是 Adaptive-RAG 存在的理由。

## Step 10 · 換你組一個

現在把積木全部給你。改下面的 `policy` 就是在設計新架構 —— 不用改任何程式碼。

### 練習題

1. **Rewrite-Retrieve-Read**（Ma et al. 2023）：先改寫 query 再檢索，不做評估。
2. **IRCoT**（Trivedi et al. 2023）：推理和檢索交錯 —— 想一步、查一步、再想一步。
3. **FLARE**（Jiang et al. 2023）：先寫草稿，遇到沒把握的句子才針對那句去檢索。
4. **省錢版 CRAG**：最多只准檢索兩次，第二次還不行就直接說查不到。

下面示範第 1 題：

In [17]:
# 自己組一個架構 = 建一個 Architecture 物件。這是整本 notebook 唯一要「寫」的東西，
# 而且你會發現寫的全是設定，沒有一行程式邏輯。
MY_ARCH = M.Architecture(
    name="Rewrite-Retrieve-Read",                                             # 給人看的名稱，會出現在 compare 表格
    paper="Ma et al. 2023《Query Rewriting for Retrieval-Augmented LLMs》",   # 出處，純說明
    orchestration="Linear（Pre-retrieval 改寫）",                              # 編排類型，純說明
    modules=["search"],                       # 只給最基本的檢索。這個清單決定 agent「拿得到哪些工具」
    # policy 就是編排規則，會接在 BASE_POLICY 後面變成 system prompt。
    # 寫法要像規則不像建議：用「先…再…」「不要…」這種祈使句，模型才會照走。
    policy="""照 Rewrite → Retrieve → Read 三步走：

1. **Rewrite**：先把使用者的問題改寫成「文件作者可能會用的說法」。
   明確寫出改寫前後，並說明你換掉了哪些詞、為什麼。
2. **Retrieve**：用改寫後的 query 做一次 search。
3. **Read**：根據結果作答。

不要評估、不要重試 —— 這個架構刻意只在檢索「之前」下功夫，
用來對照 CRAG 那種在檢索「之後」補救的做法。""",
    max_turns=6,                              # 三步走頂多幾輪，給 6 就夠；也順便防止它自己加戲
)

# 跟前面完全一樣的跑法。arun 不知道、也不在乎這是不是現成架構。
my_run = await M.arun(MY_ARCH, QUESTION, ix)
print("\n【答案】\n" + my_run.answer[:900])

  [1] search({"query": "權限設定 deny 拒絕規則 阻擋工具執行 PreToolUse hook 封鎖", "method": "hybrid", "k": 8})
       → 8 筆


  · ## 1. Rewrite（改寫）

**改寫前**：「那個會擋東西的功能怎麼設？」

**改寫後**：「權限設定 deny 拒絕規則 阻擋工具執行 PreToolUse hook 封鎖」

**換了什麼、為什麼**：
- 「會擋東西的功能」是口語描述，文件作者不會這樣寫 → 換成技術圈的正式說法：



【答案】
## 3. Read（作答）

「會擋東西」在知識庫裡對應到**兩套互相獨立的機制**，都寫在同一個 `settings.json`，但互不干涉 — 文件明講搞混這兩個是新手最常見的誤解 [02-permissions.md#0]。先選對哪一種，再看怎麼設。

### 先選：Permission 還是 Hook？

| | Hook | Permission |
|---|---|---|
| 判斷方式 | 執行一支腳本，看 exit code | 比對規則字串 |
| 能不能做複雜判斷 | 可以（任意程式邏輯） | 不行（只有模式比對） |
| 速度 | 要開行程，較慢 | 極快 |
| 會不會問使用者 | 不會，直接擋 | 可以設成詢問 |

[02-permissions.md#0]

簡單說：**規則寫得出來就用 Permission（快），要跑邏輯判斷才用 Hook**。

---

### 做法 A：Permissions（設定規則字串）

分成 `allow` / `deny` / `ask` 三個區塊 [02-permissions.md#1]：

```json
{
  "permissions": {
    "allow": [
      "Bash(git status)",
      "Bash(git diff:*)",
      "Read(**)"
    ],
    "deny": [
      "Bash(rm -rf:*)",
      "Read(.env)",
      "Write(/Users/**/.ssh/**)"
    ],
    "ask": [
      "Ba…"
    ]
  }
}
```

[02-permissions.md#1]（`ask` 區塊的內容在檢索片段中被截斷，完整範例請看原文件）

---

### 做法 B：PreToolUse Hook（跑腳本判斷）

`PreToolUse` 在工具真正執行**之前**觸發，典型用途就是阻止危險操作 [01-hooks.md#1]


In [18]:
# 三個放一起比：什麼都不做（Naive）vs 只在檢索前下功夫（Rewrite）vs 在檢索後補救（CRAG）
print(M.compare([naive_run, my_run, crag_run]))

架構                     工具呼叫  耗時     出處數  答案長度  
─────────────────────  ────  ─────  ───  ──────
Naive RAG              1     27.9s  8    1941 字
Rewrite-Retrieve-Read  1     36.8s  8    2395 字
CRAG（Corrective RAG）   4     43.4s  7    2588 字


## Step 11 · 完全自由：Modular RAG

`MODULAR_DIY` 把七個模組全部給 agent，讓它**自己決定編排**。

這是 Modular RAG 的終極型態，也是目前 agentic RAG 的實際發展方向：
不是人事先選好架構，是 agent 看到問題當場組一個。

In [19]:
# MODULAR_DIY 的 modules 是全部七個（外加 WebSearch），policy 不規定順序，
# 只要求它開工前先說計畫、中途改計畫要明講。
# 觀察軌跡：看它自己挑了哪幾個模組、順序是什麼 —— 每次跑可能不一樣，這是正常的。
modular_run = await M.arun(M.MODULAR_DIY, "要怎麼把一個 Python 函式變成 agent 能用的工具？", ix)
print("\n【答案】\n" + modular_run.answer[:1000])

  [1] list_corpus({})
       → 41 片段 / 10 檔案


  [2] multi_search({"queries": ["如何把 Python 函式定義成 agent 工具", "tool 裝飾器 @tool 用法", "工具的 input schema 參數定義", "把工具註冊給 agent 使用", "fu)
       → 8 筆


  · **我的計畫（Branching → Conditional）**

這題是「怎麼做 X」的操作型問題，可能同時牽涉裝飾器寫法、schema 定義、註冊方式，屬於多子問題型。所以我打算：

1. `list_corpus` — 先確認知識庫範圍，判斷這題在不在射程內
2. `multi_search


  [3] grade_documents({"question": "怎麼把一個 Python 函式變成 agent 能用的工具", "chunk_ids": ["06-mcp.md#2", "06-mcp.md#1", "10-troubleshooting.)
       → 取回 4 塊完整內文


  [4] expand({"chunk_id": "06-mcp.md#2", "window": 1})
       → 取回 3 塊完整內文


  · 核心命中 `06-mcp.md#2` 的 snippet 在程式碼中間被切斷，而這題答案就在那段 code 裡 — 走「條件分支：片段不完整 → expand」，同時把候選丟進 grade 讀全文自評。



【答案】
**自評結果**：`06-mcp.md#2` = 2 分（直接回答，含完整程式碼）；`06-mcp.md#1`、`10-troubleshooting.md#1`、`02-permissions.md#3` = 1 分（相關踩雷補充）。走 CRAG 的 **CORRECT** 分支，不需要再檢索，也不必上網。

---

## 結論：用 `@tool` 裝飾器包起來，再用 `create_sdk_mcp_server` 做成 in-process MCP server

用 Agent SDK 時不需要真的開一個 server 行程，可以把 Python 函式直接包成工具、跑在同一個程序裡 [06-mcp.md#2]。

### 步驟 1：定義工具

```python
from claude_agent_sdk import tool, create_sdk_mcp_server

@tool("add", "把兩個數字相加", {
    "type": "object",
    "properties": {"a": {"type": "number"}, "b": {"type": "number"}},
    "required": ["a", "b"],
})
async def add(args):
    return {"content": [{"type": "text", "text": str(args["a"] + args["b"])}]}

server = create_sdk_mcp_server("calc", "1.0.0", [add])
```

`@tool` 三個參數依序是：工具名、描述、JSON schema；函式是 `async`，收一個 `args` dict，回傳 `{"content": [...]}` 格式 [06-mcp.md#2]。

### 步驟 2：掛到 options 上

```python
ClaudeAgentOptions(
    mcp_servers={"calc": server},
    allowed_tools=["mcp__calc__add"],
    strict_mcp_config=True,      # 忽略專案的 .mcp.json，只用這裡

## 收尾：這本教了什麼

| 架構 | 編排 | 關鍵模組 | 什麼時候用 |
|---|---|---|---|
| Naive | Linear | search | 基準線 / 語料乾淨且問題明確 |
| RAG-Fusion | Branching | multi_search | 問法多變、用詞不確定 |
| HyDE | Linear（前置轉換） | hyde_search | 問題抽象、**需要向量檢索** |
| CRAG | Conditional | grade_documents | 語料可能沒有答案，不能亂編 |
| Self-RAG | Looping | grade + 反思政策 | 正確性 > 成本 |
| Adaptive | Conditional（路由） | 分類 → 三條路 | 問題複雜度落差大 |
| Modular | 自適應 | 全部 | 你也不知道會遇到什麼 |

### 三句話總結

1. **模組是能力，編排是政策。** 名字不同的架構，多半只是政策不同。
2. **Naive RAG 答錯不是模型笨，是你沒給它承認失敗的工具。**
3. **在 Claude Agent SDK 裡，換架構 = 換一份 `Architecture`，SDK 呼叫方式一行都不用改。**

---

## 卡點對照表

| 卡點 | 原因 | 處理 |
|---|---|---|
| HyDE 效果很差 | 沒有向量索引，退回 BM25 | 跑 `uv run python index_corpus.py`（不加 `--no-vectors`） |
| Self-RAG 跑很久（90 秒以上） | Looping 編排本來就慢 | 調低 `max_turns`，或改用 CRAG |
| agent 沒照 policy 的分支走 | policy 寫得像建議不像規則 | 用「**一定要**」「**先…再…**」這種祈使句，並要求它明講選了哪支 |
| 每個架構結果都差不多 | 問題太簡單，分不出差異 | 用刻意口語化 / 超出語料範圍的問題 |
| `chunk_id 不存在` | 模型自己編了 id | 正常現象，工具會回錯誤讓它自己修 —— 這也是給學生看的重點 |
| 換了架構但行為沒變 | 改了 policy 卻沒重新 `arun` | notebook 要重跑那格 |

---

## 進階閱讀

- Modular RAG — Gao et al. 2024, arXiv:2407.21059
- CRAG — Yan et al. 2024, arXiv:2401.15884
- Self-RAG — Asai et al. 2024, ICLR
- Adaptive-RAG — Jeong et al. 2024, NAACL
- HyDE — Gao et al. 2022, arXiv:2212.10496
- Claude Agent SDK — https://code.claude.com/docs/en/agent-sdk

_配套：`01_agent_sdk_basics.ipynb` · `03_retrieval_internals.ipynb` · `../README.md`_